In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_parquet("../data/cars_clean.parquet")

df["age"] = 2023 - df["year"]
df["miles_per_year"] = df["mileage"] / df["age"].clip(lower=1)

In [2]:
features = ["manufacturer", "model", "age", "mileage", "miles_per_year",
    "drivetrain", "fuel_type", "transmission_type",
    "accidents_or_damage", "one_owner", "personal_use_only",
    "mpg_city", "mpg_hwy", "driver_rating", "driver_reviews_num",
    "seller_rating", "seller_rating_missing"]

cat_cols = ["manufacturer", "model", "drivetrain", "fuel_type", "transmission_type",
            "accidents_or_damage", "one_owner", "personal_use_only"]

df[cat_cols] = df[cat_cols].astype("category")

In [3]:
train, test = train_test_split(df, test_size=0.2, random_state=42)
train_fit, valid = train_test_split(train, test_size=0.1, random_state=42)
print(train_fit.shape, valid.shape, test.shape)

(539357, 22) (59929, 22) (149822, 22)


In [5]:
def score(name, pred):
    err = np.abs(pred - test["price"])
    return {"model": name,
            "MAE": err.mean(),
            "MAPE": (err / test["price"]).mean(),
            "MedianAE": err.median()}

results = []
y_train = np.log(train["price"])
num_cols = [c for c in features if c not in cat_cols]
cat_feats = [c for c in features if c in cat_cols]

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

prep = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), cat_feats),
])
ridge = make_pipeline(prep, Ridge(alpha=1.0))
ridge.fit(train[features], y_train)
results.append(score("Ridge", np.exp(ridge.predict(test[features]))))

In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OrdinalEncoder

rf_prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_feats),
])
rf = make_pipeline(rf_prep, RandomForestRegressor(n_estimators=200, min_samples_leaf=5,
                                                  n_jobs=-1, random_state=42))
sample = train.sample(100_000, random_state=42)
rf.fit(sample[features], np.log(sample["price"]))
results.append(score("Random Forest (100k sample)", np.exp(rf.predict(test[features]))))

In [8]:
from catboost import CatBoostRegressor

def to_cb(d):
    d = d[features].copy()
    for c in cat_feats:
        d[c] = d[c].astype(str)
    return d

cb = CatBoostRegressor(iterations=3000, learning_rate=0.08, depth=8,
                       loss_function="RMSE", random_seed=42, verbose=500)
cb.fit(to_cb(train_fit), np.log(train_fit["price"]),
       cat_features=cat_feats,
       eval_set=(to_cb(valid), np.log(valid["price"])),
       early_stopping_rounds=100)
results.append(score("CatBoost", np.exp(cb.predict(to_cb(test)))))

0:	learn: 0.5346776	test: 0.5357268	best: 0.5357268 (0)	total: 704ms	remaining: 35m 11s
500:	learn: 0.1370294	test: 0.1392783	best: 0.1392783 (500)	total: 3m 57s	remaining: 19m 44s
1000:	learn: 0.1260001	test: 0.1307935	best: 0.1307913 (999)	total: 7m 52s	remaining: 15m 43s
1500:	learn: 0.1205425	test: 0.1273380	best: 0.1273380 (1500)	total: 11m 55s	remaining: 11m 54s
2000:	learn: 0.1171993	test: 0.1255738	best: 0.1255738 (2000)	total: 16m 2s	remaining: 8m
2500:	learn: 0.1142511	test: 0.1241204	best: 0.1241198 (2498)	total: 20m 22s	remaining: 4m 3s
2999:	learn: 0.1121074	test: 0.1232339	best: 0.1232339 (2999)	total: 24m 30s	remaining: 0us

bestTest = 0.1232339202
bestIteration = 2999



In [9]:
import lightgbm as lgb

lgbm = lgb.LGBMRegressor(n_estimators=5000, learning_rate=0.05, num_leaves=127,
                         min_child_samples=20, subsample=0.8, subsample_freq=1,
                         colsample_bytree=0.8, random_state=42, verbose=-1)
lgbm.fit(train_fit[features], np.log(train_fit["price"]),
         eval_set=[(valid[features], np.log(valid["price"]))],
         callbacks=[lgb.early_stopping(100), lgb.log_evaluation(500)])
results.append(score("LightGBM", np.exp(lgbm.predict(test[features]))))

c:\Users\Manci\Desktop\GitHub projects\used-car-deal-detector\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[500]	valid_0's l2: 0.014723
[1000]	valid_0's l2: 0.0144629
[1500]	valid_0's l2: 0.0143468
Early stopping, best iteration is:
[1798]	valid_0's l2: 0.0143111


In [10]:
pd.DataFrame(results).sort_values("MAPE").round(3)

,model,MAE,MAPE,MedianAE
3,LightGBM,2239.083,0.078,1431.899
2,CatBoost,2491.806,0.083,1537.164
1,Random Forest (100k sample),3117.984,0.105,1864.089
0,Ridge,3339.921,0.112,1945.286


LightGBM had the lowest error on every metric (MAPE 7.8%), ahead of CatBoost (8.3%). Linear regression performed worse than the baseline in dollar terms, because depreciation varies strongly by model, which a linear model can't capture. Random Forest was trained on a 100k sample for speed, so its result is a lower bound. LightGBM is selected as the final model.